# Final Project: Topic Modeling
**Problem Statement:**

You are to develop your own use case for topic modeling. The use case should involve text data that interests you and should be realistic for a data scientist or machine learning developer. 

In [1]:
import pandas as pd # data manipulation
import langdetect  # language detection
import matplotlib.pyplot  # plotting
import nltk  # natural language processing
import numpy  # arrays and matrices
import pandas  # dataframes
import pyLDAvis  # plotting
import pyLDAvis.lda_model  # plotting
import regex  # regular expressions
import sklearn  # machine learning
import unicodedata  # unicode data manipulation
import random # random number generation

# Text preprocessing and feature extraction
from sklearn.decomposition import NMF  # NMF model
from sklearn.decomposition import LatentDirichletAllocation  # LDA model
from sklearn.feature_extraction.text import TfidfVectorizer # TF-IDF vectorizer
from nltk.stem import WordNetLemmatizer  # lemmatizer
from sklearn.feature_extraction.text import CountVectorizer  # Count vectorizer

# Download necessary NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
url = "https://raw.githubusercontent.com/Hunteracademic/Unsupervised_assignment_1/master/tripadvisor_hotel_reviews.csv"
df = pd.read_csv(url)
df.head()

,Review,Rating
0,nice hotel expensive parking got good deal sta...,4
1,ok nothing special charge diamond member hilto...,2
2,nice rooms not 4* experience hotel monaco seat...,3
3,"unique, great stay, wonderful time hotel monac...",5
4,"great stay great stay, went seahawk game aweso...",5


Executive Summary: In 180 to 200 words, provide an overview of the notebooks you developed. Describe the use case, data, preprocessing steps, model development, and main points of the analysis. State which model works best or that none of the models were satisfactory and provide reasons. Describe the topics and explain how the model will address the use case, or if none of the models worked well, state what the next steps should be.

Preprocessing: Clean and prepare text for LDA and NMF topic modeling. Include steps such as case normalization, lemmatization, stop word removal, and tokenization. 

### Language Filter

In [3]:
def do_language_identifying(txt):
    try: the_language = langdetect.detect(txt)
    except: the_language = 'none'
    return the_language

df['Language'] = df['Review'].apply(do_language_identifying)
df['Language'].value_counts()

Language
en    20472
fr        9
af        5
ro        2
es        1
da        1
nl        1
Name: count, dtype: int64

Removing non-english reviews

In [4]:
reviews_en = df[df['Language'] == 'en']

### Tokenization / Removing Punctuation / Case Normalization

In [5]:
WORD_RE = regex.compile(r"(?V1)\p{L}+(?:[’'-]\p{L}+)*")

def tokenize_for_topics(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = regex.sub(r"[‘’`´]", "'", text)      # normalize apostrophes
    text = regex.sub(r"[‐‑‒–—−]", "-", text)    # normalize dash variants
    return WORD_RE.findall(text)

reviews_en["Tokens"] = reviews_en["Review"].apply(tokenize_for_topics)

reviews_en['Tokens'][0]

C:\Users\Hunter\AppData\Local\Temp\ipykernel_30160\4241627905.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_en["Tokens"] = reviews_en["Review"].apply(tokenize_for_topics)


['nice',
 'hotel',
 'expensive',
 'parking',
 'got',
 'good',
 'deal',
 'stay',
 'hotel',
 'anniversary',
 'arrived',
 'late',
 'evening',
 'took',
 'advice',
 'previous',
 'reviews',
 'did',
 'valet',
 'parking',
 'check',
 'quick',
 'easy',
 'little',
 'disappointed',
 'non-existent',
 'view',
 'room',
 'room',
 'clean',
 'nice',
 'size',
 'bed',
 'comfortable',
 'woke',
 'stiff',
 'neck',
 'high',
 'pillows',
 'not',
 'soundproof',
 'like',
 'heard',
 'music',
 'room',
 'night',
 'morning',
 'loud',
 'bangs',
 'doors',
 'opening',
 'closing',
 'hear',
 'people',
 'talking',
 'hallway',
 'maybe',
 'just',
 'noisy',
 'neighbors',
 'aveda',
 'bath',
 'products',
 'nice',
 'did',
 'not',
 'goldfish',
 'stay',
 'nice',
 'touch',
 'taken',
 'advantage',
 'staying',
 'longer',
 'location',
 'great',
 'walking',
 'distance',
 'shopping',
 'overall',
 'nice',
 'experience',
 'having',
 'pay',
 'parking',
 'night']

### Removing Stop Words

In [6]:
list_stop_words = nltk.corpus.stopwords.words("English")
reviews_en['Tokens'] = reviews_en['Tokens'].apply(lambda tokens: [token for token in tokens if token not in list_stop_words])

C:\Users\Hunter\AppData\Local\Temp\ipykernel_30160\2692143640.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_en['Tokens'] = reviews_en['Tokens'].apply(lambda tokens: [token for token in tokens if token not in list_stop_words])


### Lemmatization

In [7]:
lemmatizer = WordNetLemmatizer()
reviews_en['Tokens'] = reviews_en['Tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

C:\Users\Hunter\AppData\Local\Temp\ipykernel_30160\1502571546.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_en['Tokens'] = reviews_en['Tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])


In [8]:
reviews_en['Tokens'].head()

0    [nice, hotel, expensive, parking, got, good, d...
1    [ok, nothing, special, charge, diamond, member...
2    [nice, room, experience, hotel, monaco, seattl...
3    [unique, great, stay, wonderful, time, hotel, ...
4    [great, stay, great, stay, went, seahawk, game...
Name: Tokens, dtype: object

Models: Develop code to first vectorize your text data, and then train at least six LDA and six NMF topic models on these vectors. Use clear section headings for each type of model. Record each set of hyperparameters (for both vectorization and the topic models) that you try, and find the perplexity, word-topic table, and document-topic table for each model. Present this information neatly and use it to select your best LDA and NMF models.

### Vectorizing the text data: LDA
### Hyperparameter Stack:
.....

In [9]:
LDA_data = reviews_en.copy()
LDA_data['clean_text'] = LDA_data['Tokens'].apply(lambda tokens: ' '.join(tokens))
LDA_data[['Tokens', 'clean_text']].head()

,Tokens,clean_text
0,"[nice, hotel, expensive, parking, got, good, d...",nice hotel expensive parking got good deal sta...
1,"[ok, nothing, special, charge, diamond, member...",ok nothing special charge diamond member hilto...
2,"[nice, room, experience, hotel, monaco, seattl...",nice room experience hotel monaco seattle good...
3,"[unique, great, stay, wonderful, time, hotel, ...",unique great stay wonderful time hotel monaco ...
4,"[great, stay, great, stay, went, seahawk, game...",great stay great stay went seahawk game awesom...


In [10]:
number_features = 2000

cv = CountVectorizer(
    max_df=0.95,        
    min_df=2,           
    max_features=number_features
)
 
dtm_cv = cv.fit_transform(LDA_data['clean_text'])


In [11]:
print(dtm_cv[0])

  (0, 1132)	5
  (0, 842)	2
  (0, 618)	1
  (0, 1229)	3
  (0, 752)	1
  (0, 750)	1
  (0, 455)	1
  (0, 1672)	2
  (0, 66)	1
  (0, 89)	1
  (0, 943)	1
  (0, 593)	1
  (0, 1795)	1
  (0, 29)	1
  (0, 1325)	1
  (0, 1458)	1
  (0, 1883)	1
  (0, 303)	1
  (0, 1371)	1
  (0, 552)	1
  (0, 977)	1
  (0, 501)	1
  (0, 1143)	1
  (0, 1892)	1
  (0, 1478)	3
  :	:
  (0, 1098)	1
  (0, 998)	1
  (0, 517)	1
  (0, 1184)	1
  (0, 794)	1
  (0, 1247)	1
  (0, 1741)	1
  (0, 774)	1
  (0, 1048)	1
  (0, 1142)	1
  (0, 150)	1
  (0, 1341)	1
  (0, 1801)	1
  (0, 1737)	1
  (0, 27)	1
  (0, 1674)	1
  (0, 992)	1
  (0, 987)	1
  (0, 758)	1
  (0, 1912)	1
  (0, 510)	1
  (0, 1565)	1
  (0, 1204)	1
  (0, 619)	1
  (0, 1242)	1


In [12]:
# Initialize the model with default settings
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)

# Fit the model to your Count Vectorized data
lda_model.fit(dtm_cv)

LatentDirichletAllocation(n_components=5, random_state=42)

In [ ]:
# Get the total number of features (words)
vocab_size = len(cv.get_feature_names_out())
print(f"Vocabulary Size: {vocab_size}")

# Grab 10 random words to verify the Vectorizer worked as expected
print("Random samples from vocabulary:")
feature_names = cv.get_feature_names_out()
for i in range(10):
    random_word_id = random.randint(0, vocab_size - 1)
    print(f"- {feature_names[random_word_id]}")

Vocabulary Size: 2000
Random samples from vocabulary:
- resturant
- well
- restaurant
- fab
- admit
- basis
- like
- let
- resort
- beverage


In [14]:
# Loop through each topic in lda_model.components_
for index, topic in enumerate(lda_model.components_):
    print(f'THE TOP 10 WORDS FOR TOPIC #{index}')
    
    # Get indices of the top 10 words (highest values in the component array)
    top_word_indices = topic.argsort()[-10:]
    
    # Map indices back to actual words
    top_words = [feature_names[i] for i in top_word_indices]
    print(top_words)
    print('\n')

THE TOP 10 WORDS FOR TOPIC #0
['night', 'clean', 'breakfast', 'stay', 'staff', 'good', 'location', 'great', 'room', 'hotel']


THE TOP 10 WORDS FOR TOPIC #1
['food', 'area', 'good', 'nice', 'restaurant', 'room', 'beach', 'great', 'pool', 'hotel']


THE TOP 10 WORDS FOR TOPIC #2
['bed', 'day', 'desk', 'time', 'staff', 'service', 'night', 'stay', 'hotel', 'room']


THE TOP 10 WORDS FOR TOPIC #3
['room', 'day', 'people', 'staff', 'good', 'time', 'food', 'great', 'beach', 'resort']


THE TOP 10 WORDS FOR TOPIC #4
['nice', 'like', 'bar', 'food', 'beach', 'pool', 'good', 'water', 'day', 'room']




In [15]:
# Generate topic probabilities
topic_results = lda_model.transform(dtm_cv)

# Assign topic index to the dataframe
LDA_data['Topic'] = topic_results.argmax(axis=1)

# Reorder columns and drop 'clean_text' (by omission)
LDA_data = LDA_data[['Review', 'Rating', 'Topic', 'Tokens']]

# Preview results
LDA_data.head()

,Review,Rating,Topic,Tokens
0,nice hotel expensive parking got good deal sta...,4,2,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,2,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,2,"[nice, room, experience, hotel, monaco, seattl..."
3,"unique, great stay, wonderful time hotel monac...",5,0,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,2,"[great, stay, great, stay, went, seahawk, game..."


In [16]:
# See the average rating for each topic
print(LDA_data.groupby('Topic')['Rating'].mean())

# See how many reviews fell into each topic
print(LDA_data['Topic'].value_counts())

Topic
0    4.333633
1    4.321767
2    2.835973
3    4.093281
4    2.999165
Name: Rating, dtype: float64
Topic
0    11117
2     3725
3     2530
1     1902
4     1198
Name: count, dtype: int64


In [60]:
LDA_data = reviews_en.copy()
LDA_data['clean_text'] = LDA_data['Tokens'].apply(lambda tokens: ' '.join(tokens))
LDA_data[['Tokens', 'clean_text']].head()

,Tokens,clean_text
0,"[nice, hotel, expensive, parking, got, good, d...",nice hotel expensive parking got good deal sta...
1,"[ok, nothing, special, charge, diamond, member...",ok nothing special charge diamond member hilto...
2,"[nice, room, experience, hotel, monaco, seattl...",nice room experience hotel monaco seattle good...
3,"[unique, great, stay, wonderful, time, hotel, ...",unique great stay wonderful time hotel monaco ...
4,"[great, stay, great, stay, went, seahawk, game...",great stay great stay went seahawk game awesom...


In [61]:
cv_hw = CountVectorizer(
    max_df=0.85,        
    min_df=5,           
    max_features=2000
)
 
dtm_cv_hw = cv_hw.fit_transform(LDA_data['clean_text'])

In [62]:
lda_model_hw_1 = LatentDirichletAllocation(n_components=5,
                                        max_iter=10,
                                        doc_topic_prior = 0.1,
                                        topic_word_prior = 0.01,
                                        random_state=42)

lda_model_hw_2 = LatentDirichletAllocation(n_components=5,
                                        max_iter=15,
                                        doc_topic_prior = 0.2,
                                        topic_word_prior = 0.2,
                                        random_state=42)

lda_model_hw_1.fit(dtm_cv_hw)
lda_model_hw_2.fit(dtm_cv_hw)

LatentDirichletAllocation(doc_topic_prior=0.2, max_iter=15, n_components=5,
                          random_state=42, topic_word_prior=0.2)

In [63]:
models = [lda_model_hw_1, lda_model_hw_2]

for model_num, model in enumerate(models):
    print(f"=== Model {model_num + 1} ===")
    for index, topic in enumerate(model.components_):
        print(f'THE TOP 10 WORDS FOR TOPIC #{index}')
        
        top_word_indices = topic.argsort()[-10:]
        
        top_words = [feature_names[idx] for idx in top_word_indices]
        print(top_words)
        print('\n')

=== Model 1 ===
THE TOP 10 WORDS FOR TOPIC #0
['resturant', 'airport', 'day', 'nicer', 'area', 'good', 'free', 'breakfast', 'roomy', 'hour']


THE TOP 10 WORDS FOR TOPIC #1
['nightly', 'stay', 'breakfast', 'clean', 'staff', 'good', 'lock', 'great', 'roomy', 'hour']


THE TOP 10 WORDS FOR TOPIC #2
['staff', 'service', 'time', 'desk', 'bed', 'day', 'stay', 'nightly', 'hour', 'roomy']


THE TOP 10 WORDS FOR TOPIC #3
['time', 'lock', 'view', 'service', 'stayed', 'staff', 'stay', 'great', 'roomy', 'hour']


THE TOP 10 WORDS FOR TOPIC #4
['resturant', 'roomy', 'time', 'day', 'good', 'great', 'poor', 'food', 'respect', 'beach']


=== Model 2 ===
THE TOP 10 WORDS FOR TOPIC #0
['resturant', 'day', 'car', 'good', 'area', 'free', 'nicer', 'breakfast', 'roomy', 'hour']


THE TOP 10 WORDS FOR TOPIC #1
['nightly', 'stay', 'breakfast', 'clean', 'staff', 'great', 'good', 'lock', 'roomy', 'hour']


THE TOP 10 WORDS FOR TOPIC #2
['told', 'service', 'desk', 'time', 'bed', 'day', 'stay', 'nightly', 'hour'

In [64]:
for model_num, model in enumerate(models):
    # Generate topic probabilities
    topic_results = model.transform(dtm_cv_hw)

    # Assign topic index to a copy so each model's results are preserved
    temp_df = LDA_data.copy()
    temp_df['Topic'] = topic_results.argmax(axis=1)

    # Reorder columns
    temp_df = temp_df[['Review', 'Rating', 'Topic', 'Tokens']]

    # Preview results
    print(f"=== Model {model_num + 1} Document-Topic Table ===")
    display(temp_df.head())

=== Model 1 Document-Topic Table ===


,Review,Rating,Topic,Tokens
0,nice hotel expensive parking got good deal sta...,4,2,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,2,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,2,"[nice, room, experience, hotel, monaco, seattl..."
3,"unique, great stay, wonderful time hotel monac...",5,3,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,2,"[great, stay, great, stay, went, seahawk, game..."


=== Model 2 Document-Topic Table ===


,Review,Rating,Topic,Tokens
0,nice hotel expensive parking got good deal sta...,4,2,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,2,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,2,"[nice, room, experience, hotel, monaco, seattl..."
3,"unique, great stay, wonderful time hotel monac...",5,3,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,2,"[great, stay, great, stay, went, seahawk, game..."


In [65]:
for model_num, model in enumerate(models):
    temp_df = LDA_data.copy()
    temp_df['Topic'] = model.transform(dtm_cv_hw).argmax(axis=1)

    print(f"=== Model {model_num + 1} ===")
    print(f"Perplexity: {model.perplexity(dtm_cv_hw):.4f}")
    print()
    print(temp_df.groupby('Topic')['Rating'].mean())
    print()
    print(temp_df['Topic'].value_counts())
    print('\n')

=== Model 1 ===
Perplexity: 733.3823

Topic
0    4.021616
1    4.238470
2    2.296136
3    4.669454
4    3.877304
Name: Rating, dtype: float64

Topic
1    6353
3    5143
4    3798
2    3235
0    1943
Name: count, dtype: int64


=== Model 2 ===
Perplexity: 724.9857

Topic
0    4.042691
1    4.239003
2    2.300521
3    4.684811
4    3.879054
Name: Rating, dtype: float64

Topic
1    6297
3    4997
4    3762
2    3261
0    2155
Name: count, dtype: int64




### Vectorization for NMF
### Hyperparameter Stack:
.....

In [17]:
# Create a dedicated NMF dataframe from your english reviews
nmf_data = reviews_en.copy()

# Ensure the clean_text column exists (joining the lemmatized tokens)
nmf_data['clean_text'] = nmf_data['Tokens'].apply(lambda tokens: ' '.join(tokens))

# Preview the clean data
nmf_data[['Review', 'clean_text']].head()

,Review,clean_text
0,nice hotel expensive parking got good deal sta...,nice hotel expensive parking got good deal sta...
1,ok nothing special charge diamond member hilto...,ok nothing special charge diamond member hilto...
2,nice rooms not 4* experience hotel monaco seat...,nice room experience hotel monaco seattle good...
3,"unique, great stay, wonderful time hotel monac...",unique great stay wonderful time hotel monaco ...
4,"great stay great stay, went seahawk game aweso...",great stay great stay went seahawk game awesom...


In [18]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,        
    min_df=2,           
    max_features=2000   
    )

dtm_tfidf = tfidf_vectorizer.fit_transform(nmf_data['clean_text'])

In [19]:
print(dtm_tfidf[0])

  (0, 1242)	0.10767043626422011
  (0, 619)	0.09058387508764927
  (0, 1204)	0.09487514816645559
  (0, 1565)	0.10586843210987393
  (0, 510)	0.1061168823038769
  (0, 1912)	0.09337265129287586
  (0, 758)	0.049052044622574724
  (0, 987)	0.057388186775295336
  (0, 992)	0.14889724145553188
  (0, 1674)	0.09889994008665937
  (0, 27)	0.15643079476434493
  (0, 1737)	0.13238924035936692
  (0, 1801)	0.12540482174311937
  (0, 1341)	0.16698199542783287
  (0, 150)	0.118551887012523
  (0, 1142)	0.125137535179799
  (0, 1048)	0.12953116842742557
  (0, 774)	0.1519468317604857
  (0, 1741)	0.1610350824084234
  (0, 1247)	0.07891829874566073
  (0, 794)	0.1260491851920509
  (0, 1184)	0.1745459959759628
  (0, 517)	0.09324391086661513
  (0, 998)	0.14158002774848225
  (0, 1098)	0.09138634672963757
  :	:
  (0, 1478)	0.10633608057974264
  (0, 1892)	0.07825414463883007
  (0, 1143)	0.1310274353896901
  (0, 501)	0.11739497427490733
  (0, 977)	0.07720139807798414
  (0, 552)	0.10252401799749342
  (0, 1371)	0.12873475728

In [20]:
# Initialize the NMF model
nmf_model = NMF(n_components=5, random_state=42, init='nndsvd')

# Fit the model to your Count Vectorized data
nmf_model.fit(dtm_tfidf)

# Get the vocabulary names
nmf_feature_names = tfidf_vectorizer.get_feature_names_out()

# Display the Top 10 words for each NMF topic
for index, topic in enumerate(nmf_model.components_):
    print(f'THE TOP 10 WORDS FOR NMF TOPIC #{index}')
    top_word_indices = topic.argsort()[-10:]
    print([nmf_feature_names[i] for i in top_word_indices])
    print('\n')

THE TOP 10 WORDS FOR NMF TOPIC #0
['lovely', 'paris', 'recommend', 'excellent', 'best', 'service', 'stayed', 'stay', 'staff', 'hotel']


THE TOP 10 WORDS FOR NMF TOPIC #1
['drink', 'restaurant', 'good', 'people', 'time', 'day', 'pool', 'food', 'beach', 'resort']


THE TOP 10 WORDS FOR NMF TOPIC #2
['stay', 'bathroom', 'day', 'view', 'check', 'desk', 'floor', 'night', 'bed', 'room']


THE TOP 10 WORDS FOR NMF TOPIC #3
['room', 'nice', 'minute', 'clean', 'hotel', 'location', 'station', 'breakfast', 'walk', 'good']


THE TOP 10 WORDS FOR NMF TOPIC #4
['stayed', 'clean', 'room', 'helpful', 'friendly', 'place', 'staff', 'stay', 'location', 'great']




In [21]:
# Transform the numerical matrix to get topic weights
# This shows how much each document belongs to each of the 5 topics
nmf_topic_results = nmf_model.transform(dtm_tfidf)

# Assign the winning topic index to the dataframe
nmf_data['Topic'] = nmf_topic_results.argmax(axis=1)

# Reorder columns and fix the naming to match your notebook's case-sensitivity
nmf_data = nmf_data[['Review', 'Rating', 'Topic', 'Tokens']]

# Preview results
nmf_data.head()

,Review,Rating,Topic,Tokens
0,nice hotel expensive parking got good deal sta...,4,2,"[nice, hotel, expensive, parking, got, good, d..."
1,ok nothing special charge diamond member hilto...,2,2,"[ok, nothing, special, charge, diamond, member..."
2,nice rooms not 4* experience hotel monaco seat...,3,2,"[nice, room, experience, hotel, monaco, seattl..."
3,"unique, great stay, wonderful time hotel monac...",5,4,"[unique, great, stay, wonderful, time, hotel, ..."
4,"great stay great stay, went seahawk game aweso...",5,2,"[great, stay, great, stay, went, seahawk, game..."


In [22]:
# See the average rating for each topic
print(nmf_data.groupby('Topic')['Rating'].mean())

# See how many reviews fell into each topic
print(nmf_data['Topic'].value_counts()) 

Topic
0    4.320590
1    3.881621
2    3.199848
3    4.172796
4    4.545377
Name: Rating, dtype: float64
Topic
2    5274
3    4242
1    4097
4    3537
0    3322
Name: count, dtype: int64
